# Suite PRT — Printing

The standards print any Ccpp node: a translation unit as a file, anything else as the text it stands for. Trees
built by hand print as source text without help: parentheses are added where precedence needs them, and around
declarators that would otherwise bind wrongly. Printing validates the tree and checks it against the standard first.

In [ ]:
from mbse.Programs.Ccpp import Ccpp20, Syntax as S
from mbse.Programs.Ccpp._Standard import CcppStandard
from mbse.Programs.Framework.Errors import PrintError
from support import raises

CPP26, C23 = CcppStandard(2026), CcppStandard(2023, "C")
P = CPP26.print


def n(spelling):
    return S.Identifier(spelling=spelling)


def e(spelling):
    return S.IdExpression(name=n(spelling))


def i(value):
    return S.IntegerLiteral(spelling=str(value))


def b(left, operator, right):
    return S.BinaryExpression(left=left, operator=operator, right=right)


def prim(*keywords):
    return [S.PrimitiveTypeSpecifier(keyword=k) for k in keywords]


def t(*keywords, declarator=None):
    return S.TypeId(specifiers=prim(*keywords), declarator=declarator)


def d(spelling):
    return S.IdDeclarator(name=n(spelling))


def var(declarator, *keywords, initializer=None):
    return S.SimpleDeclaration(specifiers=prim(*keywords or ("int",)),
                               declarators=[S.InitDeclarator(declarator=declarator, initializer=initializer)])


def block(*items):
    return S.CompoundStatement(items=list(items))


def stmt(expression):
    return S.ExpressionStatement(expression=expression)

## PRT-01 · Precedence: operands are parenthesized where they bind less than their place needs

In [ ]:
assert P(b(b(e("a"), "+", e("b")), "*", e("c"))) == "(a + b) * c"
assert P(b(e("a"), "-", b(e("b"), "-", e("c")))) == "a - (b - c)"
assert P(b(b(e("a"), "-", e("b")), "-", e("c"))) == "a - b - c"
def assign():
    return S.AssignmentExpression(left=e("a"), operator="=", right=S.AssignmentExpression(left=e("b"), operator="=",
                                                                                          right=e("c")))


assert P(assign()) == "a = b = c"
assert P(S.AssignmentExpression(left=assign(), operator="+=", right=i(1))) == "(a = b = c) += 1"
conditional = S.ConditionalExpression(condition=assign(), consequence=b(e("x"), ",", e("y")), alternative=assign())
assert P(conditional) == "(a = b = c) ? x, y : a = b = c"
assert P(S.CallExpression(function=b(e("f"), "+", e("g")), arguments=[b(e("a"), ",", e("b")), t("int")])) == \
    "(f + g)((a, b), int)"
assert P(S.UnaryExpression(operator="-", operand=S.UnaryExpression(operator="-", operand=e("x")))) == "- -x"
assert P(S.UnaryExpression(operator="-", operand=i(-1))) == "- -1"
assert P(S.UnaryExpression(operator="!", operand=S.UnaryExpression(operator="-", operand=e("x")))) == "!-x"
assert P(S.UnaryExpression(operator="*", operand=S.PostfixExpression(operand=e("p"), operator="++"))) == "*p++"
assert P(S.PostfixExpression(operand=S.UnaryExpression(operator="*", operand=e("p")), operator="++")) == "(*p)++"
assert P(S.MemberExpression(object=b(e("a"), "+", e("b")), operator=".", member=n("c"))) == "(a + b).c"
assert P(S.CastExpression(type=t("int"), operand=b(e("a"), "*", e("b")))) == "(int)(a * b)"
assert P(S.SizeofExpression(operand=b(e("a"), "+", e("b")))) == "sizeof(a + b)"
assert P(S.SizeofExpression(operand=e("a"))) == "sizeof a"
assert P(S.DeleteExpression(operand=b(e("a"), "+", e("b")))) == "delete (a + b)"
assert P(b(e("a"), ".*", b(e("b"), "+", e("c")))) == "a.*(b + c)"
assert P(S.ThrowExpression(operand=b(e("a"), ",", e("b")))) == "throw (a, b)"
assert P(S.FoldExpression(left=b(e("a"), "+", e("b")), operator="+")) == "((a + b) + ...)"
assert P(S.PackExpansion(pattern=b(e("a"), "+", e("b")))) == "(a + b)..."
template = S.TemplateId(name=n("X"), arguments=[b(i(1), ">", i(2)), b(i(1), "<", i(2)), b(e("a"), ",", e("b")),
                                                 S.ParenthesizedExpression(expression=b(i(3), ">>", i(1)))])
assert P(S.IdExpression(name=template)) == "X<(1 > 2), 1 < 2, (a, b), (3 >> 1)>"
call = S.CallExpression(function=e("f"))
def c_t():
    return S.IdExpression(name=S.TemplateId(name=n("C"), arguments=[t("int")]))


assert P(S.TemplateDeclaration(requires=b(b(c_t(), "||", c_t()), "&&", call), declaration=var(d("x")))) == \
    "template <> requires (C<int> || C<int>) && (f())\nint x;"
print("ok")

## PRT-02 · Declarators: inside out, with parentheses where a pointer binds less than a function or array

In [ ]:
pointer_to_function = S.FunctionDeclarator(declarator=S.PointerDeclarator(declarator=d("f")),
                                           parameters=[S.ParameterDeclaration(specifiers=prim("int"))])
assert P(var(pointer_to_function)) == "int (*f)(int);"
assert P(var(S.PointerDeclarator(declarator=S.FunctionDeclarator(declarator=d("f"))))) == "int *f();"
assert P(var(S.ArrayDeclarator(declarator=S.PointerDeclarator(declarator=d("a")), size=i(3)))) == "int (*a)[3];"
assert P(var(S.ReferenceDeclarator(declarator=S.ArrayDeclarator(declarator=d("r"), size=i(3))))) == "int &r[3];"
assert P(var(S.ArrayDeclarator(declarator=S.ReferenceDeclarator(rvalue=True, declarator=d("r"))))) == "int (&&r)[];"
qualified = S.PointerDeclarator(qualifiers=[S.CvQualifier(keyword="const")], declarator=S.PointerDeclarator(
    attributes=[S.StandardAttributeSpecifier(attributes=[S.Attribute(name="a")])], declarator=d("p")))
assert P(var(qualified)) == "int *const *[[a]] p;"
assert P(var(S.ReferenceDeclarator(attributes=[S.StandardAttributeSpecifier()], declarator=d("r")))) == "int &[[]] r;"
assert P(var(S.PointerDeclarator(qualifiers=[S.CvQualifier(keyword="volatile")]))) == "int *volatile;"
assert P(var(S.PointerDeclarator(scope=n("C"), declarator=d("m")))) == "int C::*m;"
assert P(t("int", declarator=S.FunctionDeclarator(declarator=S.PointerDeclarator(),
                                                  parameters=[S.ParameterDeclaration(specifiers=prim("char"))]))) == \
    "int(*)(char)"
assert P(t("int", declarator=S.ReferenceDeclarator(declarator=S.ArrayDeclarator(size=i(2))))) == "int&[2]"
assert P(t("int", declarator=S.ArrayDeclarator(declarator=S.ReferenceDeclarator(), size=i(2)))) == "int(&)[2]"
assert P(t("char", declarator=S.PackDeclarator())) == "char..."
assert P(S.ParameterDeclaration(specifiers=prim("int"), declarator=S.PackDeclarator(declarator=d("xs")))) == \
    "int... xs"
assert P(S.ParameterDeclaration(this_keyword=True, specifiers=[S.NamedTypeSpecifier(name=n("Self"))],
                                declarator=S.ReferenceDeclarator(rvalue=True, declarator=d("self")))) == "this Self &&self"
assert P(S.SimpleDeclaration(specifiers=[S.PlaceholderTypeSpecifier()], declarators=[S.InitDeclarator(
    declarator=S.FunctionDeclarator(declarator=d("f"), ref_qualifier="&&", qualifiers=[S.CvQualifier(keyword="const")],
                                    exception=S.NoexceptSpecifier(condition=S.BooleanLiteral(value=True)),
                                    trailing_return=t("int")))])) == \
    "auto f() const && noexcept(true) -> int;"
assert P(S.TypeId(specifiers=prim("int"))) == "int"
print("ok")

## PRT-03 · Every kind prints on its own, as the text it stands for

In [ ]:
cases = [
    (S.OperatorName(operator="new[]"), "operator new[]"), (S.OperatorName(operator="co_await"), "operator co_await"),
    (S.OperatorName(operator="+"), "operator+"), (S.ConversionName(type=t("int")), "operator int"),
    (S.LiteralOperatorName(suffix="_km"), 'operator""_km'), (S.DestructorName(type=n("T")), "~T"),
    (S.DestructorName(type=S.DecltypeSpecifier(expression=e("x"))), "~decltype(x)"),
    (S.QualifiedName(global_scope=True, qualifiers=[n("a"), S.TemplateId(name=n("b"), arguments=[i(1)])],
                     name=S.TemplateId(template_keyword=True, name=n("c"))), "::a::b<1>::template c<>"),
    (S.CharacterLiteral(prefix="U", text="x"), "U'x'"), (S.StringLiteral(text="s"), '"s"'),
    (S.RawStringLiteral(text="r"), 'R"(r)"'), (S.RawStringLiteral(prefix="L", delimiter="d", text="r"), 'LR"d(r)d"'),
    (S.UserDefinedLiteral(literal=S.FloatingLiteral(spelling="1.5"), suffix="_m"), "1.5_m"),
    (S.ConcatenatedString(parts=[S.StringLiteral(text="a"), e("B")]), '"a" B'),
    (S.BooleanLiteral(), "false"), (S.NullptrLiteral(), "nullptr"), (S.ThisExpression(), "this"),
    (S.ParenthesizedExpression(expression=e("x")), "(x)"),
    (S.DefaultCapture(mode="&"), "&"), (S.SimpleCapture(name=n("x"), pack=True), "x..."),
    (S.ThisCapture(copy=True), "*this"),
    (S.InitCapture(by_reference=True, pack=True, name=n("x"), initializer=S.InitializerList(items=[i(1)])), "&...x{1}"),
    (S.RequiresExpression(), "requires {}"),
    (S.SimpleRequirement(expression=e("x")), "x;"), (S.TypeRequirement(name=n("T")), "typename T;"),
    (S.CompoundRequirement(expression=e("x")), "{ x };"),
    (S.CompoundRequirement(expression=e("x"), noexcept=True, return_type=n("C")), "{ x } noexcept -> C;"),
    (S.NestedRequirement(constraint=e("C")), "requires C;"),
    (S.PackIndexingExpression(pack=n("xs"), index=i(0)), "xs...[0]"),
    (S.ReflectExpression(), "^^::"), (S.ReflectExpression(operand=n("std")), "^^std"),
    (S.ReflectExpression(operand=t("int")), "^^int"), (S.ReflectExpression(operand=b(e("a"), "+", e("b"))), "^^(a + b)"),
    (S.SpliceExpression(reflection=e("r")), "[: r :]"),
    (S.SpliceExpression(template_keyword=True, reflection=e("r"), arguments=[t("int")]), "template [: r :]<int>"),
    (S.SubscriptExpression(object=e("m"), indices=[i(1), i(2)]), "m[1, 2]"),
    (S.FunctionalCastExpression(type=S.PlaceholderTypeSpecifier(), initializer=S.ParenthesizedInitializer(
        arguments=[e("x")])), "auto(x)"),
    (S.TypeidExpression(operand=t("int")), "typeid(int)"), (S.AwaitExpression(operand=e("x")), "co_await x"),
    (S.SizeofPackExpression(pack=n("Ts")), "sizeof...(Ts)"),
    (S.AlignofExpression(keyword="__alignof__", operand=e("x")), "__alignof__(x)"),
    (S.NoexceptExpression(operand=e("x")), "noexcept(x)"),
    (S.NewExpression(parenthesized_type=True, type=t("int"), global_scope=True, placement=[e("p")]),
     "::new (p) (int)"),
    (S.DeleteExpression(array=True, global_scope=True, operand=e("p")), "::delete[] p"),
    (S.ThrowExpression(), "throw"), (S.YieldExpression(operand=e("x")), "co_yield x"),
    (S.PackExpansion(pattern=t("int")), "int..."),
    (S.InitializerList(trailing_comma=True, items=[i(1)]), "{1,}"),
    (S.DesignatedInitializer(designators=[S.FieldDesignator(name=n("x")), S.IndexDesignator(index=i(1), last=i(2))],
                             initializer=S.InitializerList()), ".x[1 ... 2]{}"),
    (S.FieldDesignator(name=n("y")), ".y"), (S.IndexDesignator(index=i(0)), "[0]"),
    (S.StatementExpression(body=block()), "({})"), (S.ExtensionExpression(operand=e("x")), "__extension__ x"),
    (S.DefinedExpression(name=n("X")), "defined(X)"),
    (S.DeclSpecifier(keyword="static"), "static"), (S.ExplicitSpecifier(), "explicit"),
    (S.CvQualifier(keyword="const"), "const"), (S.TypenameSpecifier(name=n("T")), "typename T"),
    (S.PlaceholderTypeSpecifier(constraint=n("C"), decltype=True), "C decltype(auto)"),
    (S.TypeofSpecifier(keyword="__typeof__", operand=t("int")), "__typeof__(int)"),
    (S.PackIndexingSpecifier(pack=n("Ts"), index=i(1)), "Ts...[1]"),
    (S.SpliceSpecifier(typename_keyword=True, reflection=e("r")), "typename [: r :]"),
    (S.ClassSpecifier(key="struct"), "struct"), (S.EnumSpecifier(key="enum", body=S.EnumeratorList()), "enum {}"),
    (S.BaseSpecifier(attributes=[S.StandardAttributeSpecifier()], type=n("B")), "[[]] B"),
    (S.IdDeclarator(name=n("x")), "x"), (S.StructuredBindingDeclarator(bindings=[d("a"), S.PackDeclarator(
        declarator=d("r"))]), "[a, ...r]"), (S.ParenthesizedDeclarator(declarator=d("x")), "(x)"),
    (S.EllipsisParameter(), "..."),
    (S.TypeParameter(), "typename"), (S.TypeParameter(constraint=n("C"), pack=True, name=n("Ts")), "C... Ts"),
    (S.TypeParameter(key="class", name=n("T"), default=t("int")), "class T = int"),
    (S.TemplateTemplateParameter(key="concept", requires=e("B"), name=n("C"), default=n("D"),
                                 parameters=[S.TypeParameter()]), "template <typename> requires B concept C = D"),
    (S.EqualInitializer(value=i(1)), "= 1"), (S.ParenthesizedInitializer(arguments=[i(1), i(2)]), "(1, 2)"),
    (S.NoexceptSpecifier(), "noexcept"), (S.ThrowSpecifier(), "throw()"),
    (S.PreconditionSpecifier(predicate=e("x")), "pre(x)"),
    (S.PostconditionSpecifier(attributes=[S.StandardAttributeSpecifier()], result=n("r"), predicate=e("r")),
     "post [[]](r: r)"), (S.PostconditionSpecifier(predicate=e("x")), "post(x)"),
    (S.StandardAttributeSpecifier(using_namespace="gnu", attributes=[S.Attribute(name="a", pack=True), S.Annotation(
        value=i(1), pack=True)]), "[[using gnu: a..., =1...]]"),
    (S.Attribute(namespace="gnu", name="cold", arguments=[i(1), t("int")]), "gnu::cold(1, int)"),
    (S.Annotation(value=i(2)), "=2"), (S.AlignasSpecifier(keyword="alignas", operand=e("Ts"), pack=True),
                                       "alignas(Ts...)"),
    (S.GnuAttributeSpecifier(keyword="__attribute", attributes=[S.Attribute(name="pure")]), "__attribute((pure))"),
    (S.DeclspecSpecifier(attributes=[S.Attribute(name="a"), S.Attribute(name="b")]), "__declspec(a b)"),
    (S.InitDeclarator(declarator=d("x"), bitfield=i(3), initializer=S.InitializerList()), "x : 3{}"),
    (S.MemberInitializer(member=n("Bs"), initializer=S.ParenthesizedInitializer(arguments=[e("x")]), pack=True),
     "Bs(x)..."),
    (S.VirtSpecifier(keyword="final"), "final"), (S.DefaultedBody(), "= default;"), (S.DeletedBody(), "= delete;"),
    (S.DeletedBody(reason=S.StringLiteral(text="why")), '= delete("why");'),
    (S.MemberList(), "{}"), (S.MemberList(items=[var(d("x"))]), "{\n    int x;\n}"),
    (S.EnumeratorList(enumerators=[S.Enumerator(name=n("A"))]), "{\n    A\n}"),
    (S.Enumerator(name=n("A"), attributes=[S.StandardAttributeSpecifier()], value=i(1)), "A [[]] = 1"),
    (S.NamespaceName(inline=True, name=n("v1")), "inline v1"), (S.NamespaceName(name=n("v1")), "v1"),
    (S.UsingDeclarator(typename_keyword=True, name=n("T"), pack=True), "typename T..."),
    (S.Handler(parameter=S.EllipsisParameter(), body=block()), "catch (...) {}"),
    (S.AsmOperand(constraint=S.StringLiteral(text="r"), value=e("x")), '"r" (x)'),
    (S.GenericAssociation(value=i(1)), "default: 1"), (S.GenericAssociation(type=t("int"), value=i(1)), "int: 1"),
    (S.Comment(text=" c"), "// c"), (S.Comment(block=True, text=" c "), "/* c */"),
    (S.LambdaDeclarator(), "()"),
    (S.LambdaDeclarator(contracts=[S.PreconditionSpecifier(predicate=e("x"))], requires=e("C"),
                        attributes=[S.StandardAttributeSpecifier()]), "() [[]] requires C pre(x)"),
]
for node, expected in cases:
    standard = next(s for s in (CPP26, C23, CcppStandard(2017)) if not s.check(node))  # C in C, removed in C++17
    printed = standard.print(node)
    assert printed == expected, (node.KIND, printed, expected)
printed_kinds = {type(node) for node, _ in cases}
print(len(printed_kinds), "kinds on their own")

## PRT-04 · Statements and declarations: layout, and the constructs no parser writes yet

In [ ]:
assert P(S.IfStatement(consteval=True, negated=True, consequence=block(), alternative=block())) == \
    "if !consteval {} else {}"
assert P(S.IfStatement(consteval=True, consequence=block())) == "if consteval {}"
assert P(S.IfStatement(condition=e("a"), consequence=S.BreakStatement(), alternative=S.IfStatement(
    condition=e("b"), consequence=S.BreakStatement()))) == "if (a)\n    break;\nelse if (b)\n    break;"
assert P(S.DoStatement(body=S.ExpressionStatement(), condition=e("x"))) == "do\n    ;\nwhile (x);"
assert P(S.ForStatement(body=block())) == "for (;;) {}"
assert P(S.ForStatement(condition=var(d("x"), initializer=S.EqualInitializer(value=i(1))), body=block())) == \
    "for (; int x = 1;) {}"
assert P(S.RangeForStatement(template_keyword=True, declaration=S.SimpleDeclaration(
    specifiers=[S.PlaceholderTypeSpecifier()], declarators=[S.InitDeclarator(declarator=d("x"))]),
    range=S.InitializerList(items=[i(1)]), body=block())) == "template for (auto x : {1}) {}"
assert P(S.ContractAssertStatement(attributes=[S.StandardAttributeSpecifier()], predicate=e("x"))) == \
    "contract_assert [[]](x);"
assert P(S.AttributedStatement(attributes=[S.StandardAttributeSpecifier(attributes=[S.Attribute(name="likely")])],
                               statement=block())) == "[[likely]] {}"
assert P(S.AttributedStatement(attributes=[S.StandardAttributeSpecifier()], statement=S.LabeledStatement(
    label=n("l")))) == "[[]] l:"
switch = S.SwitchStatement(condition=e("x"), body=block(
    S.CaseStatement(value=i(1), last=i(3), statement=stmt(e("a"))), stmt(e("b")),
    S.CaseStatement(value=i(4)), S.DefaultStatement(statement=S.DefaultStatement())))
assert P(switch) == ("switch (x) {\n    case 1 ... 3:\n        a;\n        b;\n    case 4:\n"
                     "    default:\n    default:\n}")  # labels one level in, their statements two
labelled = S.LabeledStatement(label=n("l"), statement=S.DefaultStatement(statement=var(d("x"))))
assert P(labelled) == "l:\ndefault:\nint x;"
assert P(block(labelled)) == "{\nl:\ndefault:\n    int x;\n}"  # labels are outdented
assert P(S.IfStatement(condition=e("x"), consequence=S.LabeledStatement(label=n("l"), statement=block()))) == \
    "if (x)\nl: {}"
function = S.FunctionDefinition(specifiers=prim("void"), declarator=S.FunctionDeclarator(declarator=d("f")),
                                requires=c_t(), contracts=[S.PreconditionSpecifier(predicate=e("x")),
                                                         S.PostconditionSpecifier(predicate=e("y"))],
                                body=S.DeletedBody(reason=S.StringLiteral(text="no")))
assert P(function) == 'void f() requires C<int> pre(x) post(y) = delete("no");'
declaration = S.SimpleDeclaration(specifiers=prim("void"), declarators=[S.InitDeclarator(
    declarator=S.FunctionDeclarator(declarator=d("g")), requires=c_t(), contracts=[S.PreconditionSpecifier(
        predicate=e("x"))])])
assert P(declaration) == "void g() requires C<int> pre(x);"
assert P(S.FunctionDefinition(declarator=S.FunctionDeclarator(declarator=d("S")), initializers=[S.MemberInitializer(
    member=n("x"), initializer=S.InitializerList())], body=S.DefaultedBody())) == "S() : x{} = default;"
assert P(S.FriendTypeDeclaration(types=[t("int"), S.PackExpansion(pattern=S.TypeId(specifiers=[
    S.NamedTypeSpecifier(name=n("Ts"))]))])) == "friend int, Ts...;"
assert P(S.UsingDeclaration(declarators=[S.UsingDeclarator(name=n("a")), S.UsingDeclarator(name=n("b"))])) == \
    "using a, b;"
assert P(S.ConceptDefinition(name=n("C"), attributes=[S.StandardAttributeSpecifier()], constraint=i(1))) == \
    "concept C [[]] = 1;"
assert P(S.AliasDeclaration(name=n("T"), attributes=[S.StandardAttributeSpecifier()], type=t("int"))) == \
    "using T [[]] = int;"
assert P(S.ImportDeclaration(name="m", attributes=[S.StandardAttributeSpecifier()])) == "import m [[]];"
assert P(S.ModuleDeclaration(name="m")) == "module m;"
assert P(S.UsingDirective(attributes=[S.StandardAttributeSpecifier()], name=n("std"))) == "[[]] using namespace std;"
assert P(S.LinkageSpecification(language="C", items=[var(d("x"))])) == 'extern "C" int x;'
assert P(S.AsmDeclaration(attributes=[S.StandardAttributeSpecifier()], keyword="asm", template=S.StringLiteral(
    text=""), inputs=[S.AsmOperand(name=n("x"), constraint=S.StringLiteral(text="r"), value=e("x"))])) == \
    '[[]] asm("" : : [x] "r" (x));'
assert P(S.ClassSpecifier(key="struct", attributes=[S.StandardAttributeSpecifier()], name=n("S"),
                          body=S.MemberList(items=[S.AccessSpecifier(access="public")]))) == \
    "struct [[]] S {\npublic:\n}"
assert P(S.EnumSpecifier(key="enum class", attributes=[S.StandardAttributeSpecifier()], name=n("E"),
                         base=t("int"))) == "enum class [[]] E : int"
assert P(S.SimpleDeclaration(specifiers=prim("int"), declarators=[S.InitDeclarator(bitfield=i(3))])) == "int : 3;"
assert P(S.TranslationUnit()) == "" and P(S.TranslationUnit(items=[S.EmptyDeclaration()])) == ";\n"
print("ok")

## PRT-05 · Directives, comments and enumerators

In [ ]:
enum = S.EnumeratorList(enumerators=[
    S.Comment(text=" first"), S.Enumerator(name=n("A")), S.Comment(text=" after A", trailing=True),
    S.IfdefDirective(name=n("X"), items=[S.Enumerator(name=n("B"))], alternative=S.ElseDirective(
        items=[S.Enumerator(name=n("C")), S.Comment(text=" c", trailing=True)])),
    S.Enumerator(name=n("D")), S.Comment(text=" last", block=True)])
assert P(enum) == ("{\n    // first\n    A, // after A\n#ifdef X\n    B,\n#else\n    C, // c\n#endif\n    D\n"
                   "    /* last*/\n}")
assert P(S.EnumeratorList(enumerators=[S.Comment(text=" none")])) == "{\n    // none\n}"
assert P(S.EnumeratorList(enumerators=[S.Enumerator(name=n("A"))], trailing_comma=True)) == "{\n    A,\n}"
directives = S.TranslationUnit(items=[
    S.IfDirective(condition=S.DefinedExpression(name=n("X")), items=[var(d("x"))], alternative=S.ElifdefDirective(
        negated=True, name=n("Y"), items=[], alternative=S.ElifDirective(condition=i(1), items=[]))),
    S.Comment(text=" end", trailing=True), S.DefineDirective(name=n("F"), function_like=True, variadic=True),
    S.OtherDirective(directive="pragma", text="once"), S.OtherDirective(),
    S.IncludeDirective(directive="include_next", path="a.h")])
assert P(directives) == ("#if defined(X)\nint x;\n#elifndef Y\n#elif 1\n#endif // end\n#define F(...)\n"
                         "#pragma once\n#\n#include_next \"a.h\"\n")
assert P(S.DefineDirective(name=n("G"), function_like=True)) == "#define G()"
assert P(S.IfdefDirective(name=n("X"), items=[S.Enumerator(name=n("B"))])) == "#ifdef X\nB\n#endif"  # not in a list
print("ok")

## PRT-06 · Printing validates the tree and checks it against the standard

In [ ]:
with raises(PrintError, match="a BinaryExpression needs a right"):
    P(S.BinaryExpression(left=e("a"), operator="+"))
with raises(PrintError, match="items[0]: Comment.text must be a str, got int"):
    P(S.TranslationUnit(items=[S.Comment(text=1)]))
with raises(PrintError, match="BinaryExpression.operator '<=>' needs C++20, but this is C++17"):
    CcppStandard(2017).print(b(e("a"), "<=>", e("b")))
with raises(PrintError, match="right: GenericSelection is not C++, but this is C++20"):
    Ccpp20.print(b(e("a"), "+", S.GenericSelection(controlling=i(1))))
assert C23.print(S.GenericSelection(controlling=t("int"), associations=[S.GenericAssociation(value=i(1))])) == \
    "_Generic(int, default: 1)"
assert C23.print(S.TypeofSpecifier(keyword="typeof_unqual", operand=e("x"))) == "typeof_unqual(x)"
assert C23.print(S.AtomicTypeSpecifier(type=t("int"))) == "_Atomic(int)"
assert C23.print(S.BitIntSpecifier(width=i(7))) == "_BitInt(7)"
assert C23.print(S.CompoundLiteralExpression(type=t("int"), initializer=S.InitializerList(items=[i(1)]))) == \
    "(int){1}"
print("ok")